In [1]:
!pip install langchain-text-splitters chromadb langchain-chroma langchain-openai matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 21.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 20.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 20.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 20.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 21.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 13.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [langchain-chroma][chromadb]ce-hub]


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma # this is vector db 
from langchain_openai import OpenAIEmbeddings
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
import numpy as np


In [3]:
print("loading the pdf.....")
loader = PyPDFLoader("AI_TrainingPLan.pdf")
docs = loader.load()
# till here we are only reading the document and creating document objects

loading the pdf.....


In [6]:
print(f"  Pages loaded     : {len(docs)}")
print(f"  First page meta  : {docs[0].metadata}")
print(f"  Sample content   : {docs[0].page_content[:200]}...") # extract 1st 200 characters only

  Pages loaded     : 8
  First page meta  : {'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-04T15:38:00+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-04T15:38:00+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': 'Comprehensive Generative AI Learning Path', 'trapped': '/False', 'source': 'AI_TrainingPLan.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
  Sample content   : Comprehensive Generative AI Learning Path
Curated by Anish Roychowdhury
April 4, 2025
Contents
1 F oundations of Generative AI 3
1.1 Introduction to Generative AI . . . . . . . . . . . . . . . . . . ....


In [8]:
print("----Chunking Documents------")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap = 40)

chunks = text_splitter.split_documents(docs)

print("Total chunks created...", {len(chunks)})
print("Sample of page content: ", {chunks[0].page_content[:300]})

print("Printing Sample chunks")

print(chunks[0])
print("===================")
print(chunks[1])
print("===================")
print(chunks[2])
print("===================")

----Chunking Documents------
Total chunks created... {206}
Sample of page content:  {'Comprehensive Generative AI Learning Path\nCurated by Anish Roychowdhury\nApril 4, 2025\nContents'}
Printing Sample chunks
page_content='Comprehensive Generative AI Learning Path
Curated by Anish Roychowdhury
April 4, 2025
Contents' metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-04T15:38:00+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-04T15:38:00+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': 'Comprehensive Generative AI Learning Path', 'trapped': '/False', 'source': 'AI_TrainingPLan.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
page_content='April 4, 2025
Contents
1 F oundations of Generative AI 3' metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-04T15:38:00+00:00', 'author': '', 'keywor

In [ ]:
print("-------Embedding Model----------")

embedding_model = OpenAIEmbeddings(model ="text-embedding-3-large")

print(f"Embedding {len(chunks)} ....")

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
    collection_name = "ai_training_plan"
)

print("Total chunks stored in vectorstore.....\n")
total_stored = vectorstore._collection.count()
print(f"total vectors stored in Chroma DB : {total_stored}")

-------Embedding Model----------
Embedding 206 ....
Total chunks stored in vectorstore.....

total vectors stored in Chroma DB : 206
